# Hepatocellular Carcinoma Detection Pipeline
## Production-Grade 3D Medical Imaging for Liver Cancer Segmentation & Classification

**Architecture Overview:**
- Layer 1: Multimodal Ingestion (NIfTI + Clinical Bio-tabular Fusion)
- Layer 2: Precision Preprocessing (HU Clipping, Isotropic Resampling, Augmentation)
- Layer 3: Hierarchical Cascade (Localizer → Segmentor)
- Layer 4: Diagnostic Logic (10% Malignancy Threshold)
- Layer 5: Training & Metrics (Dice + CE Loss, NPV Tracking)

In [ ]:
# ============================================================================
# CELL 0: INSTALL DEPENDENCIES (Run this first on Kaggle!)
# ============================================================================
# Install MONAI and required packages for 3D medical imaging
import subprocess
import sys

def install_packages():
    """Install required packages for HCC detection pipeline"""
    packages = [
        'monai[all]',           # MONAI with all optional dependencies
        'nibabel',              # NIfTI file handling
        'scikit-image',         # Image processing utilities
        'einops',               # Tensor operations
    ]
    
    for package in packages:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    
    print("\n✅ All packages installed successfully!")

# Run installation
install_packages()

In [ ]:
# ============================================================================
# CELL 1: IMPORTS AND CONFIGURATION
# ============================================================================
import os
import sys
import logging
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Callable, Any, Union
from dataclasses import dataclass, field
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split

# MONAI imports
import monai
from monai.config import print_config
from monai.data import (
    CacheDataset, 
    PersistentDataset,
    DataLoader as MonaiDataLoader,
    decollate_batch,
    list_data_collate
)
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, CropForegroundd, RandCropByPosNegLabeld,
    RandFlipd, RandRotate90d, RandGaussianNoised, RandAffined,
    ToTensord, EnsureTyped, AsDiscreted, Activationsd, Invertd,
    SaveImaged, SpatialPadd, CenterSpatialCropd, RandSpatialCropd,
    RandGaussianSmoothd, RandScaleIntensityd, RandShiftIntensityd,
    MapTransform, Transform
)
from monai.networks.nets import DynUNet, UNet, UNETR, SegResNet
from monai.networks.layers import Norm
from monai.losses import DiceLoss, DiceCELoss, DiceFocalLoss
from monai.metrics import DiceMetric, ConfusionMatrixMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism

# Suppress warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Print MONAI configuration
print_config()

# Set determinism for reproducibility
set_determinism(seed=42)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

ImportError: cannot import name 'compute_meandice' from 'monai.metrics' (c:\Users\asus\Downloads\HEPTACELLULAR CARCINOMA DETECTION\.venv\Lib\site-packages\monai\metrics\__init__.py)

In [2]:
# ============================================================================
# CELL 2: PIPELINE CONFIGURATION
# ============================================================================
@dataclass
class PipelineConfig:
    """
    Production-grade configuration for HCC detection pipeline.
    All hyperparameters are centralized here for reproducibility.
    """
    # Dataset paths (Kaggle datasets)
    data_paths: List[str] = field(default_factory=lambda: [
        '/kaggle/input/cbct-liver-and-liver-tumor-segmentation-train-data',
        '/kaggle/input/liver-tumor-segmentation-part-2',
        '/kaggle/input/liver-tumor-segmentation'
    ])
    
    # Output directories
    output_dir: str = './output'
    cache_dir: str = './cache'
    model_dir: str = './models'
    log_dir: str = './logs'
    
    # Image preprocessing
    hu_min: float = -75.0          # HU clipping lower bound (liver window)
    hu_max: float = 150.0          # HU clipping upper bound (liver window)
    target_spacing: Tuple[float, float, float] = (0.8, 0.8, 0.8)  # 0.8mm isotropic
    
    # ROI crop sizes
    localizer_roi_size: Tuple[int, int, int] = (96, 96, 96)    # Stage A: Low-res
    segmentor_roi_size: Tuple[int, int, int] = (128, 128, 128)  # Stage B: High-res
    
    # Data augmentation
    rotation_range: float = 40.0   # ±40° rotation
    flip_prob: float = 0.5
    noise_std: float = 0.1
    
    # Training hyperparameters
    batch_size: int = 2
    num_workers: int = 4
    max_epochs: int = 300
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    
    # Data split (65/10/25 patient-level)
    train_ratio: float = 0.65
    val_ratio: float = 0.10
    test_ratio: float = 0.25
    
    # Model architecture
    localizer_channels: Tuple[int, ...] = (32, 64, 128, 256)
    segmentor_channels: Tuple[int, ...] = (32, 64, 128, 256, 512)
    num_classes: int = 3  # Background, Liver, Tumor
    
    # Diagnostic thresholds
    malignancy_threshold: float = 0.10  # 10% voxel threshold
    
    # Mixed precision training
    use_amp: bool = True
    
    # Memory management
    cache_rate: float = 0.5  # Cache 50% of data in memory
    
    def __post_init__(self):
        """Create output directories"""
        for dir_path in [self.output_dir, self.cache_dir, self.model_dir, self.log_dir]:
            Path(dir_path).mkdir(parents=True, exist_ok=True)

# Initialize configuration
config = PipelineConfig()
logger.info(f"Pipeline Configuration: {config}")

NameError: name 'logger' is not defined

In [ ]:
# ============================================================================
# CELL 3: LAYER 1 - MULTIMODAL INGESTION (NIfTI + Clinical Bio-tabular Fusion)
# ============================================================================

class ClinicalDataProcessor:
    """
    Processes clinical bio-tabular data (AFP levels, etc.) for late-stage fusion.
    Handles missing values and normalization for neural network consumption.
    """
    
    CLINICAL_FEATURES = [
        'afp_level',           # Alpha-fetoprotein (ng/mL)
        'age',                 # Patient age
        'gender',              # 0: Female, 1: Male  
        'cirrhosis',           # Cirrhosis presence (0/1)
        'hepatitis_b',         # Hepatitis B status (0/1)
        'hepatitis_c',         # Hepatitis C status (0/1)
        'bilirubin',           # Bilirubin level (mg/dL)
        'albumin',             # Albumin level (g/dL)
        'platelet_count',      # Platelet count (10^9/L)
        'tumor_size_cm',       # Estimated tumor size from reports
    ]
    
    # Normalization parameters (from literature/training data statistics)
    FEATURE_STATS = {
        'afp_level': {'mean': 500.0, 'std': 2000.0},
        'age': {'mean': 60.0, 'std': 12.0},
        'gender': {'mean': 0.6, 'std': 0.5},
        'cirrhosis': {'mean': 0.5, 'std': 0.5},
        'hepatitis_b': {'mean': 0.3, 'std': 0.46},
        'hepatitis_c': {'mean': 0.2, 'std': 0.4},
        'bilirubin': {'mean': 1.2, 'std': 1.5},
        'albumin': {'mean': 3.5, 'std': 0.6},
        'platelet_count': {'mean': 150.0, 'std': 75.0},
        'tumor_size_cm': {'mean': 3.0, 'std': 2.5},
    }
    
    def __init__(self):
        self.feature_dim = len(self.CLINICAL_FEATURES)
    
    def process(self, clinical_data: Optional[Dict[str, float]]) -> torch.Tensor:
        """
        Process clinical data into normalized tensor.
        Missing values are imputed with population means.
        
        Args:
            clinical_data: Dictionary of clinical features or None
            
        Returns:
            Normalized clinical feature tensor of shape (feature_dim,)
        """
        if clinical_data is None:
            # Return zeros for missing clinical data (will be masked in fusion)
            return torch.zeros(self.feature_dim, dtype=torch.float32)
        
        features = []
        for feat_name in self.CLINICAL_FEATURES:
            value = clinical_data.get(feat_name, self.FEATURE_STATS[feat_name]['mean'])
            # Z-score normalization
            normalized = (value - self.FEATURE_STATS[feat_name]['mean']) / \
                        (self.FEATURE_STATS[feat_name]['std'] + 1e-8)
            features.append(normalized)
        
        return torch.tensor(features, dtype=torch.float32)


class MultimodalLiverDataset(Dataset):
    """
    Layer 1: Multimodal Ingestion Dataset
    
    Handles volumetric .nii files and maps them to clinical bio-tabular vectors
    for late-stage fusion. Supports multiple dataset sources (LiTS17, TCGA-LIHC, etc.)
    
    Features:
    - Patient-level organization for proper data splitting
    - Clinical data integration (AFP, demographics, etc.)
    - Memory-efficient lazy loading with optional caching
    """
    
    def __init__(
        self,
        data_dicts: List[Dict[str, str]],
        transform: Optional[Callable] = None,
        clinical_data: Optional[Dict[str, Dict[str, float]]] = None,
        cache_rate: float = 0.0,
        cache_dir: Optional[str] = None
    ):
        """
        Args:
            data_dicts: List of dicts with 'image', 'label', 'patient_id' keys
            transform: MONAI Compose transform pipeline
            clinical_data: Dict mapping patient_id to clinical features
            cache_rate: Fraction of data to cache in memory (0-1)
            cache_dir: Directory for persistent caching
        """
        self.data_dicts = data_dicts
        self.transform = transform
        self.clinical_data = clinical_data or {}
        self.clinical_processor = ClinicalDataProcessor()
        self.cache_rate = cache_rate
        self.cache_dir = cache_dir
        
        # Build patient index for patient-level operations
        self._build_patient_index()
        
        logger.info(f"Initialized MultimodalLiverDataset with {len(self.data_dicts)} samples")
        logger.info(f"Unique patients: {len(self.patient_index)}")
    
    def _build_patient_index(self):
        """Build index mapping patient_id to sample indices"""
        self.patient_index: Dict[str, List[int]] = {}
        for idx, data_dict in enumerate(self.data_dicts):
            patient_id = data_dict.get('patient_id', f'patient_{idx}')
            if patient_id not in self.patient_index:
                self.patient_index[patient_id] = []
            self.patient_index[patient_id].append(idx)
    
    def __len__(self) -> int:
        return len(self.data_dicts)
    
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """
        Load and transform a single sample with multimodal fusion.
        
        Returns:
            Dict containing:
                - 'image': Transformed CT volume tensor
                - 'label': Segmentation mask tensor
                - 'clinical': Clinical feature vector
                - 'patient_id': Patient identifier
                - 'has_clinical': Boolean mask for clinical data availability
        """
        data = dict(self.data_dicts[idx])
        patient_id = data.get('patient_id', f'patient_{idx}')
        
        # Apply MONAI transforms (loading, preprocessing, augmentation)
        if self.transform is not None:
            data = self.transform(data)
        
        # Add clinical data for late-stage fusion
        clinical_raw = self.clinical_data.get(patient_id, None)
        data['clinical'] = self.clinical_processor.process(clinical_raw)
        data['has_clinical'] = torch.tensor(clinical_raw is not None, dtype=torch.bool)
        data['patient_id'] = patient_id
        
        return data
    
    def get_patient_ids(self) -> List[str]:
        """Get list of unique patient IDs for patient-level splitting"""
        return list(self.patient_index.keys())


def discover_nifti_files(data_paths: List[str]) -> List[Dict[str, str]]:
    """
    Discover all NIfTI files from multiple dataset sources.
    Handles various directory structures from different datasets.
    
    Args:
        data_paths: List of root directories containing NIfTI data
        
    Returns:
        List of dicts with 'image', 'label', 'patient_id' keys
    """
    data_dicts = []
    
    for root_path in data_paths:
        root = Path(root_path)
        if not root.exists():
            logger.warning(f"Path does not exist: {root_path}")
            continue
            
        # Pattern 1: LiTS-style (volume-X.nii, segmentation-X.nii)
        for vol_file in root.rglob('volume*.nii*'):
            seg_file = vol_file.parent / vol_file.name.replace('volume', 'segmentation')
            if seg_file.exists():
                patient_id = vol_file.stem.replace('volume-', '').replace('volume_', '')
                data_dicts.append({
                    'image': str(vol_file),
                    'label': str(seg_file),
                    'patient_id': f"lits_{patient_id}"
                })
        
        # Pattern 2: TCGA-style (patient folders)
        for patient_dir in root.iterdir():
            if patient_dir.is_dir():
                images = list(patient_dir.glob('*CT*.nii*')) + list(patient_dir.glob('*image*.nii*'))
                labels = list(patient_dir.glob('*seg*.nii*')) + list(patient_dir.glob('*label*.nii*'))
                if images and labels:
                    data_dicts.append({
                        'image': str(images[0]),
                        'label': str(labels[0]),
                        'patient_id': patient_dir.name
                    })
        
        # Pattern 3: IRCADb-style (nested patient folders)
        for patient_dir in root.rglob('PATIENT_*'):
            if patient_dir.is_dir():
                images = list(patient_dir.rglob('*image*.nii*'))
                labels = list(patient_dir.rglob('*label*.nii*')) + list(patient_dir.rglob('*mask*.nii*'))
                if images and labels:
                    data_dicts.append({
                        'image': str(images[0]),
                        'label': str(labels[0]),
                        'patient_id': patient_dir.name
                    })
    
    logger.info(f"Discovered {len(data_dicts)} NIfTI image-label pairs")
    return data_dicts


def patient_level_split(
    data_dicts: List[Dict[str, str]],
    train_ratio: float = 0.65,
    val_ratio: float = 0.10,
    test_ratio: float = 0.25,
    random_seed: int = 42
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """
    Patient-level data splitting to prevent data leakage.
    Ensures slices from the same patient stay in the same split.
    
    Args:
        data_dicts: List of data dictionaries
        train_ratio: Training set ratio (default 0.65)
        val_ratio: Validation set ratio (default 0.10)  
        test_ratio: Test set ratio (default 0.25)
        random_seed: Random seed for reproducibility
        
    Returns:
        Tuple of (train_dicts, val_dicts, test_dicts)
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    
    # Group by patient
    patient_to_samples: Dict[str, List[Dict]] = {}
    for d in data_dicts:
        pid = d.get('patient_id', 'unknown')
        if pid not in patient_to_samples:
            patient_to_samples[pid] = []
        patient_to_samples[pid].append(d)
    
    patients = list(patient_to_samples.keys())
    n_patients = len(patients)
    
    # Split patients (not samples)
    train_patients, test_patients = train_test_split(
        patients, 
        test_size=test_ratio, 
        random_state=random_seed
    )
    
    val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)
    train_patients, val_patients = train_test_split(
        train_patients,
        test_size=val_ratio_adjusted,
        random_state=random_seed
    )
    
    # Collect samples for each split
    train_dicts = [s for p in train_patients for s in patient_to_samples[p]]
    val_dicts = [s for p in val_patients for s in patient_to_samples[p]]
    test_dicts = [s for p in test_patients for s in patient_to_samples[p]]
    
    logger.info(f"Patient-level split:")
    logger.info(f"  Train: {len(train_patients)} patients, {len(train_dicts)} samples")
    logger.info(f"  Val: {len(val_patients)} patients, {len(val_dicts)} samples")
    logger.info(f"  Test: {len(test_patients)} patients, {len(test_dicts)} samples")
    
    return train_dicts, val_dicts, test_dicts

In [ ]:
# ============================================================================
# CELL 4: LAYER 2 - PRECISION PREPROCESSING (MONAI Transform Pipeline)
# ============================================================================

class EnsureVolumeShape(MapTransform):
    """
    Custom transform to ensure minimum volume dimensions for ROI cropping.
    Pads volumes that are smaller than the target ROI size.
    """
    
    def __init__(self, keys: List[str], min_size: Tuple[int, int, int]):
        super().__init__(keys)
        self.min_size = min_size
    
    def __call__(self, data: Dict) -> Dict:
        d = dict(data)
        for key in self.keys:
            if key in d:
                shape = d[key].shape[-3:]  # Spatial dimensions
                pad_needed = [max(0, m - s) for m, s in zip(self.min_size, shape)]
                if any(p > 0 for p in pad_needed):
                    # Calculate padding (symmetric)
                    padding = []
                    for p in pad_needed:
                        padding.extend([p // 2, p - p // 2])
                    padding = tuple(padding)
                    d[key] = F.pad(d[key], padding[::-1])  # F.pad uses reverse order
        return d


def get_localizer_transforms(config: PipelineConfig, is_train: bool = True) -> Compose:
    """
    Stage A (Localizer) Transform Pipeline - Lower resolution for liver ROI extraction.
    
    Preprocessing:
    - HU Clipping: [-75, 150] for liver window
    - Resampling: 0.8mm isotropic to preserve small lesion details
    - Orientation: RAS standard
    
    Augmentation (training only):
    - 3D Rotations: ±40°
    - Random flipping
    - Gaussian noise injection
    """
    
    base_transforms = [
        # Load NIfTI files
        LoadImaged(keys=['image', 'label'], reader='NibabelReader'),
        
        # Ensure channel dimension is first
        EnsureChannelFirstd(keys=['image', 'label']),
        
        # Standardize orientation to RAS
        Orientationd(keys=['image', 'label'], axcodes='RAS'),
        
        # Resample to 0.8mm isotropic resolution
        # Critical for detecting sub-centimeter lesions
        Spacingd(
            keys=['image', 'label'],
            pixdim=config.target_spacing,
            mode=('bilinear', 'nearest')  # Bilinear for image, nearest for labels
        ),
        
        # HU Clipping: Window intensities for liver visualization
        # -75 to 150 HU captures liver parenchyma and tumors
        ScaleIntensityRanged(
            keys=['image'],
            a_min=config.hu_min,
            a_max=config.hu_max,
            b_min=0.0,
            b_max=1.0,
            clip=True
        ),
        
        # Crop to foreground to reduce computation
        CropForegroundd(
            keys=['image', 'label'],
            source_key='image',
            margin=10
        ),
        
        # Ensure minimum volume size
        SpatialPadd(
            keys=['image', 'label'],
            spatial_size=config.localizer_roi_size,
            mode='constant'
        ),
    ]
    
    if is_train:
        # Training augmentations
        aug_transforms = [
            # Random crop with balanced pos/neg sampling
            RandCropByPosNegLabeld(
                keys=['image', 'label'],
                label_key='label',
                spatial_size=config.localizer_roi_size,
                pos=1,
                neg=1,
                num_samples=4,
                image_key='image',
                image_threshold=0
            ),
            
            # 3D Rotations: ±40 degrees
            RandAffined(
                keys=['image', 'label'],
                prob=0.5,
                rotate_range=[np.deg2rad(config.rotation_range)] * 3,
                mode=('bilinear', 'nearest'),
                padding_mode='zeros'
            ),
            
            # Random flipping along all axes
            RandFlipd(keys=['image', 'label'], prob=config.flip_prob, spatial_axis=0),
            RandFlipd(keys=['image', 'label'], prob=config.flip_prob, spatial_axis=1),
            RandFlipd(keys=['image', 'label'], prob=config.flip_prob, spatial_axis=2),
            
            # Gaussian noise injection for robustness
            RandGaussianNoised(
                keys=['image'],
                prob=0.3,
                std=config.noise_std
            ),
            
            # Intensity augmentation
            RandScaleIntensityd(keys=['image'], factors=0.1, prob=0.3),
            RandShiftIntensityd(keys=['image'], offsets=0.1, prob=0.3),
        ]
        base_transforms.extend(aug_transforms)
    else:
        # Validation: Center crop
        base_transforms.append(
            CenterSpatialCropd(
                keys=['image', 'label'],
                roi_size=config.localizer_roi_size
            )
        )
    
    # Final conversion to tensors
    base_transforms.append(EnsureTyped(keys=['image', 'label'], dtype=torch.float32))
    
    return Compose(base_transforms)


def get_segmentor_transforms(config: PipelineConfig, is_train: bool = True) -> Compose:
    """
    Stage B (Segmentor) Transform Pipeline - Higher resolution for precise tumor segmentation.
    
    Applied after ROI extraction from Stage A.
    Uses larger crop sizes for fine-grained segmentation.
    """
    
    base_transforms = [
        LoadImaged(keys=['image', 'label'], reader='NibabelReader'),
        EnsureChannelFirstd(keys=['image', 'label']),
        Orientationd(keys=['image', 'label'], axcodes='RAS'),
        
        # Higher resolution resampling for Stage B
        Spacingd(
            keys=['image', 'label'],
            pixdim=config.target_spacing,
            mode=('bilinear', 'nearest')
        ),
        
        ScaleIntensityRanged(
            keys=['image'],
            a_min=config.hu_min,
            a_max=config.hu_max,
            b_min=0.0,
            b_max=1.0,
            clip=True
        ),
        
        CropForegroundd(
            keys=['image', 'label'],
            source_key='image',
            margin=5
        ),
        
        SpatialPadd(
            keys=['image', 'label'],
            spatial_size=config.segmentor_roi_size,
            mode='constant'
        ),
    ]
    
    if is_train:
        aug_transforms = [
            RandCropByPosNegLabeld(
                keys=['image', 'label'],
                label_key='label',
                spatial_size=config.segmentor_roi_size,
                pos=2,  # More positive samples for tumor detection
                neg=1,
                num_samples=2,
                image_key='image',
                image_threshold=0
            ),
            
            RandAffined(
                keys=['image', 'label'],
                prob=0.5,
                rotate_range=[np.deg2rad(config.rotation_range)] * 3,
                scale_range=[0.1, 0.1, 0.1],
                mode=('bilinear', 'nearest'),
                padding_mode='zeros'
            ),
            
            RandFlipd(keys=['image', 'label'], prob=config.flip_prob, spatial_axis=0),
            RandFlipd(keys=['image', 'label'], prob=config.flip_prob, spatial_axis=1),
            RandFlipd(keys=['image', 'label'], prob=config.flip_prob, spatial_axis=2),
            
            RandGaussianNoised(keys=['image'], prob=0.3, std=config.noise_std),
            RandGaussianSmoothd(keys=['image'], prob=0.2, sigma_x=(0.5, 1.0)),
            RandScaleIntensityd(keys=['image'], factors=0.1, prob=0.3),
            RandShiftIntensityd(keys=['image'], offsets=0.1, prob=0.3),
        ]
        base_transforms.extend(aug_transforms)
    else:
        base_transforms.append(
            CenterSpatialCropd(
                keys=['image', 'label'],
                roi_size=config.segmentor_roi_size
            )
        )
    
    base_transforms.append(EnsureTyped(keys=['image', 'label'], dtype=torch.float32))
    
    return Compose(base_transforms)


# Post-processing transforms for inference
post_transforms = Compose([
    Activationsd(keys='pred', softmax=True),
    AsDiscreted(keys='pred', argmax=True),
])

logger.info("Transform pipelines initialized")

In [ ]:
# ============================================================================
# CELL 5: LAYER 3 - HIERARCHICAL CASCADE (Stage A: Localizer)
# ============================================================================

class ResidualBlock3D(nn.Module):
    """
    3D Residual block with optional squeeze-and-excitation.
    Used in both encoder and decoder paths.
    """
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        use_se: bool = True,
        se_ratio: int = 16
    ):
        super().__init__()
        
        self.conv1 = nn.Conv3d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.InstanceNorm3d(out_channels, affine=True)
        self.conv2 = nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.InstanceNorm3d(out_channels, affine=True)
        self.relu = nn.LeakyReLU(0.2, inplace=True)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.InstanceNorm3d(out_channels, affine=True)
            )
        
        # Squeeze-and-Excitation
        self.use_se = use_se
        if use_se:
            self.se = nn.Sequential(
                nn.AdaptiveAvgPool3d(1),
                nn.Conv3d(out_channels, out_channels // se_ratio, 1),
                nn.ReLU(inplace=True),
                nn.Conv3d(out_channels // se_ratio, out_channels, 1),
                nn.Sigmoid()
            )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        if self.use_se:
            out = out * self.se(out)
        
        out += self.shortcut(x)
        return self.relu(out)


class LiverLocalizer(nn.Module):
    """
    Stage A: Low-Resolution 3D DynUNet for Liver ROI Extraction
    
    Purpose: Quickly identify the liver region in full CT scans
    to reduce computational load for the high-resolution segmentor.
    
    Architecture:
    - Encoder: 4 levels of strided convolutions
    - Decoder: Transposed convolutions with skip connections
    - Output: Binary liver mask (liver vs background)
    """
    
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 2,  # Background + Liver
        channels: Tuple[int, ...] = (32, 64, 128, 256),
        strides: Tuple[int, ...] = (2, 2, 2, 2),
        kernel_sizes: Tuple[int, ...] = (3, 3, 3, 3),
        deep_supervision: bool = True
    ):
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.channels = channels
        self.strides = strides
        self.deep_supervision = deep_supervision
        self.num_levels = len(channels)
        
        # Use MONAI's DynUNet as backbone
        self.backbone = DynUNet(
            spatial_dims=3,
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_sizes,
            strides=strides,
            upsample_kernel_size=strides,
            filters=channels,
            norm_name='INSTANCE',
            act_name='LEAKYRELU',
            deep_supervision=deep_supervision,
            deep_supr_num=2,
            res_block=True,
            trans_bias=True
        )
        
        logger.info(f"LiverLocalizer initialized: {sum(p.numel() for p in self.parameters()):,} parameters")
    
    def forward(self, x: torch.Tensor) -> Union[torch.Tensor, List[torch.Tensor]]:
        """
        Forward pass through localizer.
        
        Args:
            x: Input CT volume [B, 1, D, H, W]
            
        Returns:
            Liver segmentation logits [B, 2, D, H, W]
            If deep_supervision: List of outputs at different resolutions
        """
        return self.backbone(x)
    
    def extract_roi(
        self,
        image: torch.Tensor,
        label: Optional[torch.Tensor] = None,
        margin: int = 10,
        min_size: Tuple[int, int, int] = (64, 64, 64)
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Tuple[slice, ...]]:
        """
        Extract liver ROI from full CT scan using model predictions.
        
        Args:
            image: Full CT volume [B, 1, D, H, W]
            label: Optional segmentation mask
            margin: Margin around detected liver
            min_size: Minimum ROI size
            
        Returns:
            Tuple of (cropped_image, cropped_label, crop_slices)
        """
        self.eval()
        with torch.no_grad():
            pred = self.forward(image)
            if isinstance(pred, list):
                pred = pred[0]
            
            # Get liver mask (class 1)
            liver_mask = torch.argmax(pred, dim=1) == 1
        
        # Find bounding box of liver region
        batch_crops = []
        for b in range(image.shape[0]):
            mask = liver_mask[b]
            if mask.any():
                # Find nonzero indices
                nonzero = torch.nonzero(mask)
                min_coords = nonzero.min(dim=0).values
                max_coords = nonzero.max(dim=0).values
                
                # Add margin and ensure minimum size
                slices = []
                for i in range(3):
                    start = max(0, min_coords[i].item() - margin)
                    end = min(image.shape[i+2], max_coords[i].item() + margin + 1)
                    
                    # Ensure minimum size
                    current_size = end - start
                    if current_size < min_size[i]:
                        diff = min_size[i] - current_size
                        start = max(0, start - diff // 2)
                        end = min(image.shape[i+2], end + diff - diff // 2)
                    
                    slices.append(slice(start, end))
                
                batch_crops.append(tuple(slices))
            else:
                # No liver found, return center crop
                batch_crops.append(tuple(
                    slice(s//2 - m//2, s//2 + m//2)
                    for s, m in zip(image.shape[2:], min_size)
                ))
        
        # For simplicity, use first sample's crop for the batch
        crop_slices = batch_crops[0]
        full_slices = (slice(None), slice(None)) + crop_slices
        
        cropped_image = image[full_slices]
        cropped_label = label[full_slices] if label is not None else None
        
        return cropped_image, cropped_label, crop_slices

In [ ]:
# ============================================================================
# CELL 6: LAYER 3 - HIERARCHICAL CASCADE (Stage B: ResUNet Segmentor)
# ============================================================================

class ClinicalFusionModule(nn.Module):
    """
    Late-stage fusion module for integrating clinical bio-tabular data
    with imaging features. Uses attention-based gating.
    """
    
    def __init__(
        self,
        clinical_dim: int,
        feature_channels: int,
        hidden_dim: int = 64
    ):
        super().__init__()
        
        self.clinical_dim = clinical_dim
        self.feature_channels = feature_channels
        
        # Clinical feature encoder
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, feature_channels),
            nn.Sigmoid()  # Attention weights
        )
        
        # Fusion gate
        self.fusion_gate = nn.Sequential(
            nn.Conv3d(feature_channels * 2, feature_channels, 1),
            nn.InstanceNorm3d(feature_channels),
            nn.Sigmoid()
        )
    
    def forward(
        self,
        image_features: torch.Tensor,
        clinical_features: torch.Tensor,
        has_clinical: torch.Tensor
    ) -> torch.Tensor:
        """
        Fuse clinical data with image features using attention.
        
        Args:
            image_features: [B, C, D, H, W] spatial features
            clinical_features: [B, clinical_dim] tabular features
            has_clinical: [B] boolean mask for clinical availability
            
        Returns:
            Fused features [B, C, D, H, W]
        """
        B, C, D, H, W = image_features.shape
        
        # Encode clinical features to channel attention weights
        clinical_attention = self.clinical_encoder(clinical_features)  # [B, C]
        clinical_attention = clinical_attention.view(B, C, 1, 1, 1)
        
        # Apply attention to image features
        attended_features = image_features * clinical_attention
        
        # Gated fusion
        concat = torch.cat([image_features, attended_features], dim=1)
        gate = self.fusion_gate(concat)
        
        # Apply mask for samples without clinical data
        has_clinical = has_clinical.view(B, 1, 1, 1, 1).float()
        fused = gate * attended_features + (1 - gate) * image_features
        output = has_clinical * fused + (1 - has_clinical) * image_features
        
        return output


class HybridResUNet(nn.Module):
    """
    Stage B: High-Resolution ResUNet with Hybrid Residual Connections
    
    Purpose: Precise segmentation of malignant voxels within liver ROI.
    
    Architecture:
    - Encoder: ResNet-style blocks with squeeze-and-excitation
    - Decoder: Transposed convolutions with dense skip connections
    - Clinical Fusion: Late-stage integration of bio-tabular data
    - Output: Multi-class segmentation (Background, Liver, Tumor)
    
    Key Features:
    - Hybrid residual connections (within-block + across-block)
    - Deep supervision for gradient flow
    - Attention-based clinical data fusion
    """
    
    def __init__(
        self,
        in_channels: int = 1,
        out_channels: int = 3,  # Background, Liver, Tumor
        channels: Tuple[int, ...] = (32, 64, 128, 256, 512),
        clinical_dim: int = 10,
        use_clinical_fusion: bool = True,
        deep_supervision: bool = True
    ):
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.channels = channels
        self.num_levels = len(channels)
        self.use_clinical_fusion = use_clinical_fusion
        self.deep_supervision = deep_supervision
        
        # Initial convolution
        self.init_conv = nn.Sequential(
            nn.Conv3d(in_channels, channels[0], 3, padding=1, bias=False),
            nn.InstanceNorm3d(channels[0], affine=True),
            nn.LeakyReLU(0.2, inplace=True)
        )
        
        # Encoder pathway
        self.encoders = nn.ModuleList()
        self.downsample = nn.ModuleList()
        for i in range(self.num_levels):
            in_ch = channels[0] if i == 0 else channels[i-1]
            out_ch = channels[i]
            
            self.encoders.append(nn.Sequential(
                ResidualBlock3D(in_ch, out_ch),
                ResidualBlock3D(out_ch, out_ch)
            ))
            
            if i < self.num_levels - 1:
                self.downsample.append(nn.Sequential(
                    nn.Conv3d(out_ch, out_ch, 3, stride=2, padding=1, bias=False),
                    nn.InstanceNorm3d(out_ch, affine=True),
                    nn.LeakyReLU(0.2, inplace=True)
                ))
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            ResidualBlock3D(channels[-1], channels[-1]),
            ResidualBlock3D(channels[-1], channels[-1])
        )
        
        # Clinical fusion module
        if use_clinical_fusion:
            self.clinical_fusion = ClinicalFusionModule(
                clinical_dim=clinical_dim,
                feature_channels=channels[-1]
            )
        
        # Decoder pathway
        self.upsample = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for i in range(self.num_levels - 1, 0, -1):
            in_ch = channels[i]
            out_ch = channels[i-1]
            
            self.upsample.append(nn.Sequential(
                nn.ConvTranspose3d(in_ch, out_ch, 4, stride=2, padding=1, bias=False),
                nn.InstanceNorm3d(out_ch, affine=True),
                nn.LeakyReLU(0.2, inplace=True)
            ))
            
            self.decoders.append(nn.Sequential(
                ResidualBlock3D(out_ch * 2, out_ch),  # Skip connection doubles channels
                ResidualBlock3D(out_ch, out_ch)
            ))
        
        # Output heads
        self.output_conv = nn.Conv3d(channels[0], out_channels, 1)
        
        # Deep supervision heads
        if deep_supervision:
            self.ds_heads = nn.ModuleList([
                nn.Conv3d(channels[i], out_channels, 1)
                for i in range(1, min(4, self.num_levels))
            ])
        
        self._init_weights()
        logger.info(f"HybridResUNet initialized: {sum(p.numel() for p in self.parameters()):,} parameters")
    
    def _init_weights(self):
        """Initialize network weights"""
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='leaky_relu')
            elif isinstance(m, nn.InstanceNorm3d):
                if m.weight is not None:
                    nn.init.constant_(m.weight, 1)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(
        self,
        x: torch.Tensor,
        clinical: Optional[torch.Tensor] = None,
        has_clinical: Optional[torch.Tensor] = None
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, List[torch.Tensor]]]:
        """
        Forward pass through segmentor.
        
        Args:
            x: Input CT ROI [B, 1, D, H, W]
            clinical: Clinical features [B, clinical_dim]
            has_clinical: Boolean mask [B]
            
        Returns:
            Segmentation logits [B, 3, D, H, W]
            If deep_supervision: Tuple of (output, deep_supervision_outputs)
        """
        # Initial convolution
        x = self.init_conv(x)
        
        # Encoder with skip connections
        skips = []
        for i in range(self.num_levels):
            x = self.encoders[i](x)
            if i < self.num_levels - 1:
                skips.append(x)
                x = self.downsample[i](x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Clinical fusion at bottleneck
        if self.use_clinical_fusion and clinical is not None:
            if has_clinical is None:
                has_clinical = torch.ones(x.shape[0], dtype=torch.bool, device=x.device)
            x = self.clinical_fusion(x, clinical, has_clinical)
        
        # Decoder with skip connections
        ds_outputs = []
        for i, (up, dec) in enumerate(zip(self.upsample, self.decoders)):
            x = up(x)
            skip = skips[-(i+1)]
            
            # Handle size mismatches
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
            # Deep supervision
            if self.deep_supervision and i < len(self.ds_heads):
                ds_outputs.append(self.ds_heads[i](x))
        
        # Final output
        output = self.output_conv(x)
        
        if self.deep_supervision and self.training:
            return output, ds_outputs
        return output


class CascadeSegmentationModel(nn.Module):
    """
    Complete Hierarchical Cascade for HCC Detection
    
    Two-stage architecture:
    - Stage A (Localizer): Extract liver ROI from full CT
    - Stage B (Segmentor): Fine-grained tumor segmentation within ROI
    
    Benefits:
    - Memory efficient: Process full scans at low-res first
    - Precision: High-res segmentation focused on relevant region
    - Clinical integration: Bio-tabular fusion for enhanced prediction
    """
    
    def __init__(self, config: PipelineConfig):
        super().__init__()
        
        self.config = config
        
        # Stage A: Liver Localizer
        self.localizer = LiverLocalizer(
            in_channels=1,
            out_channels=2,  # Background + Liver
            channels=config.localizer_channels,
            deep_supervision=True
        )
        
        # Stage B: Tumor Segmentor
        self.segmentor = HybridResUNet(
            in_channels=1,
            out_channels=config.num_classes,
            channels=config.segmentor_channels,
            clinical_dim=ClinicalDataProcessor().feature_dim,
            use_clinical_fusion=True,
            deep_supervision=True
        )
    
    def forward(
        self,
        x: torch.Tensor,
        clinical: Optional[torch.Tensor] = None,
        has_clinical: Optional[torch.Tensor] = None,
        stage: str = 'both'
    ) -> Dict[str, Any]:
        """
        Forward pass through cascade.
        
        Args:
            x: Input CT volume
            clinical: Clinical features
            has_clinical: Clinical availability mask
            stage: 'localizer', 'segmentor', or 'both'
            
        Returns:
            Dict with 'localizer_output' and/or 'segmentor_output'
        """
        outputs = {}
        
        if stage in ['localizer', 'both']:
            localizer_out = self.localizer(x)
            outputs['localizer_output'] = localizer_out
        
        if stage in ['segmentor', 'both']:
            segmentor_out = self.segmentor(x, clinical, has_clinical)
            outputs['segmentor_output'] = segmentor_out
        
        return outputs

logger.info("Cascade architecture modules defined")

In [ ]:
# ============================================================================
# CELL 7: LAYER 5 - HYBRID LOSS FUNCTION & TRAINING INFRASTRUCTURE
# ============================================================================

class HybridLoss(nn.Module):
    """
    Hybrid Loss: Dice + Cross-Entropy to address class imbalance
    
    The combination addresses:
    - Dice Loss: Handles class imbalance, focuses on overlap
    - Cross-Entropy: Provides stable gradients, per-voxel supervision
    
    Optional components:
    - Focal weighting for hard examples
    - Deep supervision loss aggregation
    """
    
    def __init__(
        self,
        num_classes: int = 3,
        dice_weight: float = 0.5,
        ce_weight: float = 0.5,
        class_weights: Optional[torch.Tensor] = None,
        include_background: bool = False,
        smooth: float = 1e-5,
        focal_gamma: float = 0.0  # Set > 0 for focal loss
    ):
        super().__init__()
        
        self.num_classes = num_classes
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight
        self.focal_gamma = focal_gamma
        
        # MONAI's DiceCELoss handles both components
        self.dice_ce_loss = DiceCELoss(
            include_background=include_background,
            to_onehot_y=True,
            softmax=True,
            smooth_nr=smooth,
            smooth_dr=smooth,
            ce_weight=class_weights,
            lambda_dice=dice_weight,
            lambda_ce=ce_weight
        )
        
        # Optional focal dice loss for harder cases
        if focal_gamma > 0:
            self.focal_dice = DiceFocalLoss(
                include_background=include_background,
                to_onehot_y=True,
                softmax=True,
                gamma=focal_gamma
            )
    
    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
        deep_supervision_preds: Optional[List[torch.Tensor]] = None,
        ds_weights: Optional[List[float]] = None
    ) -> torch.Tensor:
        """
        Compute hybrid loss with optional deep supervision.
        
        Args:
            pred: Predicted logits [B, C, D, H, W]
            target: Ground truth [B, 1, D, H, W]
            deep_supervision_preds: List of deep supervision outputs
            ds_weights: Weights for deep supervision losses
            
        Returns:
            Combined loss value
        """
        # Main loss
        if self.focal_gamma > 0:
            main_loss = self.focal_dice(pred, target)
        else:
            main_loss = self.dice_ce_loss(pred, target)
        
        # Deep supervision losses
        if deep_supervision_preds is not None and len(deep_supervision_preds) > 0:
            if ds_weights is None:
                ds_weights = [0.5 ** (i + 1) for i in range(len(deep_supervision_preds))]
            
            ds_loss = 0.0
            for i, ds_pred in enumerate(deep_supervision_preds):
                # Resize target to match prediction size
                target_resized = F.interpolate(
                    target.float(),
                    size=ds_pred.shape[2:],
                    mode='nearest'
                ).long()
                ds_loss += ds_weights[i] * self.dice_ce_loss(ds_pred, target_resized)
            
            # Combine main and deep supervision losses
            total_loss = main_loss + ds_loss
        else:
            total_loss = main_loss
        
        return total_loss


class NPVDiceMetric:
    """
    Combined metric tracker for Dice Similarity Coefficient and 
    Negative Predictive Value (NPV) for clinical triage.
    
    NPV = TN / (TN + FN)
    High NPV ensures that negative predictions are trustworthy.
    """
    
    def __init__(self, num_classes: int = 3, include_background: bool = False):
        self.num_classes = num_classes
        self.include_background = include_background
        
        # MONAI metrics
        self.dice_metric = DiceMetric(
            include_background=include_background,
            reduction='mean_batch'
        )
        
        self.confusion_metric = ConfusionMatrixMetric(
            include_background=include_background,
            metric_name=['sensitivity', 'specificity', 'precision'],
            compute_sample=True
        )
        
        self.reset()
    
    def reset(self):
        """Reset accumulated metrics"""
        self.dice_metric.reset()
        self.confusion_metric.reset()
        self.tn_sum = 0
        self.fn_sum = 0
        self.batch_npvs = []
    
    def update(self, pred: torch.Tensor, target: torch.Tensor):
        """
        Update metrics with batch predictions.
        
        Args:
            pred: Predicted segmentation (argmax applied) [B, 1, D, H, W]
            target: Ground truth [B, 1, D, H, W]
        """
        # Convert to one-hot for MONAI metrics
        pred_onehot = F.one_hot(pred.squeeze(1).long(), self.num_classes).permute(0, 4, 1, 2, 3)
        target_onehot = F.one_hot(target.squeeze(1).long(), self.num_classes).permute(0, 4, 1, 2, 3)
        
        # Update Dice
        self.dice_metric(pred_onehot, target_onehot)
        
        # Compute NPV per batch (for tumor class = 2)
        tumor_pred = (pred == 2).float()
        tumor_target = (target == 2).float()
        
        tn = ((tumor_pred == 0) & (tumor_target == 0)).sum().item()
        fn = ((tumor_pred == 0) & (tumor_target == 1)).sum().item()
        
        self.tn_sum += tn
        self.fn_sum += fn
        
        if tn + fn > 0:
            batch_npv = tn / (tn + fn)
            self.batch_npvs.append(batch_npv)
    
    def compute(self) -> Dict[str, float]:
        """
        Compute aggregated metrics.
        
        Returns:
            Dict with 'dice_liver', 'dice_tumor', 'npv', 'mean_dice'
        """
        dice_values = self.dice_metric.aggregate()
        
        results = {
            'dice_liver': dice_values[0].item() if len(dice_values) > 0 else 0.0,
            'dice_tumor': dice_values[1].item() if len(dice_values) > 1 else 0.0,
            'mean_dice': dice_values.mean().item() if len(dice_values) > 0 else 0.0,
        }
        
        # Compute NPV
        if self.tn_sum + self.fn_sum > 0:
            results['npv'] = self.tn_sum / (self.tn_sum + self.fn_sum)
        else:
            results['npv'] = 1.0  # No false negatives
        
        return results


class TrainingLogger:
    """
    Comprehensive logging for training metrics, loss curves, and checkpoints.
    """
    
    def __init__(self, log_dir: str, experiment_name: str = 'hcc_detection'):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        
        self.experiment_name = experiment_name
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        
        # Initialize log file
        self.log_file = self.log_dir / f'{experiment_name}_{self.timestamp}.log'
        
        # Metrics history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_dice': [],
            'val_dice': [],
            'val_npv': [],
            'learning_rate': []
        }
        
        # File handler for persistent logging
        file_handler = logging.FileHandler(self.log_file)
        file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
        logger.addHandler(file_handler)
        
        logger.info(f"Training logger initialized: {self.log_file}")
    
    def log_epoch(
        self,
        epoch: int,
        train_loss: float,
        val_loss: float,
        train_metrics: Dict[str, float],
        val_metrics: Dict[str, float],
        lr: float
    ):
        """Log metrics for an epoch"""
        self.history['train_loss'].append(train_loss)
        self.history['val_loss'].append(val_loss)
        self.history['train_dice'].append(train_metrics.get('mean_dice', 0))
        self.history['val_dice'].append(val_metrics.get('mean_dice', 0))
        self.history['val_npv'].append(val_metrics.get('npv', 0))
        self.history['learning_rate'].append(lr)
        
        logger.info(
            f"Epoch {epoch:03d} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
            f"Train Dice: {train_metrics.get('mean_dice', 0):.4f} | "
            f"Val Dice: {val_metrics.get('mean_dice', 0):.4f} | "
            f"Val NPV: {val_metrics.get('npv', 0):.4f} | "
            f"LR: {lr:.6f}"
        )
    
    def save_history(self):
        """Save training history to JSON"""
        history_file = self.log_dir / f'{self.experiment_name}_{self.timestamp}_history.json'
        with open(history_file, 'w') as f:
            json.dump(self.history, f, indent=2)
        logger.info(f"Training history saved to {history_file}")

In [ ]:
# ============================================================================
# CELL 8: LAYER 4 - DIAGNOSTIC LOGIC (Post-Processing & Classification)
# ============================================================================
from scipy import ndimage
from scipy.ndimage import label as scipy_label, binary_dilation

class DiagnosticProcessor:
    """
    Layer 4: Diagnostic Logic for Clinical Triage
    
    Implements the 10% malignancy threshold rule:
    - If ≥10% of a lesion's voxels are flagged as malignant, classify entire mass as cancer
    - Connected component analysis to identify individual lesions
    - Volume and morphometric feature extraction
    """
    
    def __init__(
        self,
        malignancy_threshold: float = 0.10,
        min_lesion_volume_mm3: float = 10.0,  # Minimum lesion size
        voxel_spacing: Tuple[float, float, float] = (0.8, 0.8, 0.8)
    ):
        self.malignancy_threshold = malignancy_threshold
        self.min_lesion_volume_mm3 = min_lesion_volume_mm3
        self.voxel_spacing = voxel_spacing
        self.voxel_volume_mm3 = np.prod(voxel_spacing)
    
    def process_segmentation(
        self,
        segmentation: np.ndarray,
        probability_map: Optional[np.ndarray] = None
    ) -> Dict[str, Any]:
        """
        Apply diagnostic logic to segmentation output.
        
        Args:
            segmentation: Predicted segmentation [D, H, W] with values {0, 1, 2}
                         0=background, 1=liver, 2=tumor
            probability_map: Optional soft predictions [C, D, H, W]
            
        Returns:
            Dict containing:
                - 'lesions': List of lesion info dicts
                - 'cancer_detected': Boolean
                - 'num_malignant_lesions': Count of malignant lesions
                - 'total_tumor_volume_mm3': Total tumor volume
                - 'refined_segmentation': Post-processed segmentation
        """
        # Extract tumor mask
        tumor_mask = (segmentation == 2).astype(np.int32)
        liver_mask = (segmentation == 1).astype(np.int32)
        
        # Connected component analysis for tumor regions
        labeled_tumors, num_tumors = scipy_label(tumor_mask)
        
        lesions = []
        num_malignant = 0
        total_tumor_volume = 0.0
        refined_seg = segmentation.copy()
        
        for lesion_id in range(1, num_tumors + 1):
            lesion_mask = (labeled_tumors == lesion_id)
            lesion_voxels = lesion_mask.sum()
            lesion_volume_mm3 = lesion_voxels * self.voxel_volume_mm3
            
            # Filter out small artifacts
            if lesion_volume_mm3 < self.min_lesion_volume_mm3:
                # Remove small lesions from segmentation
                refined_seg[lesion_mask] = 1  # Reclassify as liver
                continue
            
            # Calculate malignancy ratio
            if probability_map is not None:
                # Use soft predictions for malignancy assessment
                tumor_probs = probability_map[2][lesion_mask]  # Class 2 = tumor
                malignant_voxels = (tumor_probs > 0.5).sum()
                malignancy_ratio = malignant_voxels / lesion_voxels
            else:
                # All tumor voxels are considered malignant in hard segmentation
                malignancy_ratio = 1.0
            
            # Apply 10% threshold rule
            is_malignant = malignancy_ratio >= self.malignancy_threshold
            
            # Compute lesion centroid
            coords = np.where(lesion_mask)
            centroid = tuple(np.mean(c) for c in coords)
            
            # Compute lesion diameter (approximate)
            max_extent = max(c.max() - c.min() for c in coords)
            diameter_mm = max_extent * self.voxel_spacing[0]
            
            lesion_info = {
                'id': lesion_id,
                'volume_mm3': lesion_volume_mm3,
                'volume_cc': lesion_volume_mm3 / 1000,
                'diameter_mm': diameter_mm,
                'centroid_voxel': centroid,
                'malignancy_ratio': malignancy_ratio,
                'is_malignant': is_malignant,
                'voxel_count': int(lesion_voxels)
            }
            
            lesions.append(lesion_info)
            
            if is_malignant:
                num_malignant += 1
                total_tumor_volume += lesion_volume_mm3
                # Mark entire lesion as malignant in refined segmentation
                refined_seg[lesion_mask] = 2
            else:
                # Reclassify as benign/liver tissue
                refined_seg[lesion_mask] = 1
        
        return {
            'lesions': lesions,
            'cancer_detected': num_malignant > 0,
            'num_malignant_lesions': num_malignant,
            'total_tumor_volume_mm3': total_tumor_volume,
            'total_tumor_volume_cc': total_tumor_volume / 1000,
            'refined_segmentation': refined_seg,
            'num_lesions_total': len(lesions)
        }
    
    def generate_report(self, diagnostic_result: Dict[str, Any]) -> str:
        """
        Generate human-readable diagnostic report.
        
        Args:
            diagnostic_result: Output from process_segmentation
            
        Returns:
            Formatted diagnostic report string
        """
        report_lines = [
            "=" * 60,
            "HEPATOCELLULAR CARCINOMA DETECTION REPORT",
            "=" * 60,
            "",
            f"Cancer Detected: {'YES - URGENT REVIEW REQUIRED' if diagnostic_result['cancer_detected'] else 'NO'}",
            f"Total Lesions Analyzed: {diagnostic_result['num_lesions_total']}",
            f"Malignant Lesions: {diagnostic_result['num_malignant_lesions']}",
            f"Total Tumor Volume: {diagnostic_result['total_tumor_volume_cc']:.2f} cc",
            "",
            "-" * 60,
            "LESION DETAILS:",
            "-" * 60,
        ]
        
        for lesion in diagnostic_result['lesions']:
            status = "MALIGNANT" if lesion['is_malignant'] else "BENIGN/INDETERMINATE"
            report_lines.extend([
                f"\nLesion #{lesion['id']}:",
                f"  Status: {status}",
                f"  Volume: {lesion['volume_cc']:.3f} cc ({lesion['volume_mm3']:.1f} mm³)",
                f"  Max Diameter: {lesion['diameter_mm']:.1f} mm",
                f"  Malignancy Ratio: {lesion['malignancy_ratio']*100:.1f}%",
            ])
        
        report_lines.extend([
            "",
            "-" * 60,
            "CLINICAL NOTES:",
            f"  - Malignancy threshold applied: {self.malignancy_threshold*100:.0f}%",
            f"  - Minimum lesion size: {self.min_lesion_volume_mm3:.1f} mm³",
            f"  - Voxel resolution: {self.voxel_spacing} mm",
            "=" * 60,
        ])
        
        return "\n".join(report_lines)


def apply_post_processing(
    pred_logits: torch.Tensor,
    config: PipelineConfig
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Apply complete post-processing pipeline to model predictions.
    
    Args:
        pred_logits: Raw model output [B, C, D, H, W]
        config: Pipeline configuration
        
    Returns:
        Tuple of (refined_segmentation, diagnostic_results)
    """
    # Convert to probabilities
    probs = F.softmax(pred_logits, dim=1)
    
    # Get hard segmentation
    seg = torch.argmax(probs, dim=1)
    
    # Process each sample in batch
    batch_results = []
    refined_segs = []
    
    processor = DiagnosticProcessor(
        malignancy_threshold=config.malignancy_threshold,
        voxel_spacing=config.target_spacing
    )
    
    for b in range(seg.shape[0]):
        seg_np = seg[b].cpu().numpy()
        probs_np = probs[b].cpu().numpy()
        
        result = processor.process_segmentation(seg_np, probs_np)
        batch_results.append(result)
        refined_segs.append(result['refined_segmentation'])
    
    refined_segs = np.stack(refined_segs, axis=0)
    
    return refined_segs, batch_results


logger.info("Diagnostic logic module initialized")

In [ ]:
# ============================================================================
# CELL 9: TRAINING ENGINE
# ============================================================================

class CascadeTrainer:
    """
    Production-grade trainer for the hierarchical cascade model.
    
    Features:
    - Two-stage training (Localizer → Segmentor)
    - Mixed precision training with gradient scaling
    - Learning rate scheduling with warmup
    - Early stopping with patience
    - Model checkpointing
    - Comprehensive metric tracking
    """
    
    def __init__(
        self,
        config: PipelineConfig,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device = DEVICE
    ):
        self.config = config
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        # Initialize models
        self.cascade = CascadeSegmentationModel(config).to(device)
        
        # Loss functions
        # Localizer: Binary (background vs liver)
        self.localizer_loss = HybridLoss(
            num_classes=2,
            dice_weight=0.5,
            ce_weight=0.5,
            include_background=False
        )
        
        # Segmentor: Multi-class with class weighting
        tumor_weight = torch.tensor([1.0, 2.0, 5.0], device=device)  # Upweight tumor class
        self.segmentor_loss = HybridLoss(
            num_classes=3,
            dice_weight=0.5,
            ce_weight=0.5,
            class_weights=tumor_weight,
            include_background=False,
            focal_gamma=2.0  # Focal loss for hard examples
        )
        
        # Optimizers (Adam with lr=0.001 as specified)
        self.localizer_optimizer = torch.optim.Adam(
            self.cascade.localizer.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
        
        self.segmentor_optimizer = torch.optim.Adam(
            self.cascade.segmentor.parameters(),
            lr=config.learning_rate,
            weight_decay=config.weight_decay
        )
        
        # Learning rate schedulers
        self.localizer_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.localizer_optimizer,
            T_0=50,
            T_mult=2
        )
        
        self.segmentor_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.segmentor_optimizer,
            T_0=50,
            T_mult=2
        )
        
        # Mixed precision
        self.scaler = GradScaler() if config.use_amp else None
        
        # Metrics
        self.localizer_metrics = NPVDiceMetric(num_classes=2)
        self.segmentor_metrics = NPVDiceMetric(num_classes=3)
        
        # Logger
        self.logger = TrainingLogger(config.log_dir, 'hcc_cascade')
        
        # Best model tracking
        self.best_val_dice = 0.0
        self.patience_counter = 0
        self.patience = 30
        
        logger.info("CascadeTrainer initialized")
        logger.info(f"Localizer params: {sum(p.numel() for p in self.cascade.localizer.parameters()):,}")
        logger.info(f"Segmentor params: {sum(p.numel() for p in self.cascade.segmentor.parameters()):,}")
    
    def train_epoch_localizer(self, epoch: int) -> Tuple[float, Dict[str, float]]:
        """Train localizer for one epoch"""
        self.cascade.localizer.train()
        total_loss = 0.0
        self.localizer_metrics.reset()
        
        for batch_idx, batch in enumerate(self.train_loader):
            images = batch['image'].to(self.device)
            # Convert multi-class labels to binary (liver vs background)
            labels = (batch['label'] > 0).long().to(self.device)
            
            self.localizer_optimizer.zero_grad()
            
            if self.config.use_amp:
                with autocast():
                    outputs = self.cascade.localizer(images)
                    if isinstance(outputs, list):
                        loss = self.localizer_loss(outputs[0], labels, outputs[1:])
                        outputs = outputs[0]
                    else:
                        loss = self.localizer_loss(outputs, labels)
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.localizer_optimizer)
                self.scaler.update()
            else:
                outputs = self.cascade.localizer(images)
                if isinstance(outputs, list):
                    loss = self.localizer_loss(outputs[0], labels, outputs[1:])
                    outputs = outputs[0]
                else:
                    loss = self.localizer_loss(outputs, labels)
                
                loss.backward()
                self.localizer_optimizer.step()
            
            total_loss += loss.item()
            
            # Update metrics
            pred = torch.argmax(outputs, dim=1, keepdim=True)
            self.localizer_metrics.update(pred, labels)
            
            if batch_idx % 10 == 0:
                logger.info(f"  Localizer Epoch {epoch} Batch {batch_idx}/{len(self.train_loader)} Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(self.train_loader)
        metrics = self.localizer_metrics.compute()
        
        return avg_loss, metrics
    
    def train_epoch_segmentor(self, epoch: int) -> Tuple[float, Dict[str, float]]:
        """Train segmentor for one epoch"""
        self.cascade.segmentor.train()
        total_loss = 0.0
        self.segmentor_metrics.reset()
        
        for batch_idx, batch in enumerate(self.train_loader):
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            clinical = batch.get('clinical', None)
            has_clinical = batch.get('has_clinical', None)
            
            if clinical is not None:
                clinical = clinical.to(self.device)
            if has_clinical is not None:
                has_clinical = has_clinical.to(self.device)
            
            self.segmentor_optimizer.zero_grad()
            
            if self.config.use_amp:
                with autocast():
                    outputs = self.cascade.segmentor(images, clinical, has_clinical)
                    if isinstance(outputs, tuple):
                        main_out, ds_outs = outputs
                        loss = self.segmentor_loss(main_out, labels, ds_outs)
                    else:
                        loss = self.segmentor_loss(outputs, labels)
                        main_out = outputs
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.segmentor_optimizer)
                self.scaler.update()
            else:
                outputs = self.cascade.segmentor(images, clinical, has_clinical)
                if isinstance(outputs, tuple):
                    main_out, ds_outs = outputs
                    loss = self.segmentor_loss(main_out, labels, ds_outs)
                else:
                    loss = self.segmentor_loss(outputs, labels)
                    main_out = outputs
                
                loss.backward()
                self.segmentor_optimizer.step()
            
            total_loss += loss.item()
            
            # Update metrics
            pred = torch.argmax(main_out, dim=1, keepdim=True)
            self.segmentor_metrics.update(pred, labels)
            
            if batch_idx % 10 == 0:
                logger.info(f"  Segmentor Epoch {epoch} Batch {batch_idx}/{len(self.train_loader)} Loss: {loss.item():.4f}")
        
        avg_loss = total_loss / len(self.train_loader)
        metrics = self.segmentor_metrics.compute()
        
        return avg_loss, metrics
    
    @torch.no_grad()
    def validate(self, stage: str = 'segmentor') -> Tuple[float, Dict[str, float]]:
        """Validate model"""
        if stage == 'localizer':
            model = self.cascade.localizer
            loss_fn = self.localizer_loss
            metrics = self.localizer_metrics
        else:
            model = self.cascade.segmentor
            loss_fn = self.segmentor_loss
            metrics = self.segmentor_metrics
        
        model.eval()
        total_loss = 0.0
        metrics.reset()
        
        for batch in self.val_loader:
            images = batch['image'].to(self.device)
            labels = batch['label'].to(self.device)
            
            if stage == 'localizer':
                labels = (labels > 0).long()
            
            if stage == 'segmentor':
                clinical = batch.get('clinical', None)
                has_clinical = batch.get('has_clinical', None)
                if clinical is not None:
                    clinical = clinical.to(self.device)
                if has_clinical is not None:
                    has_clinical = has_clinical.to(self.device)
                outputs = model(images, clinical, has_clinical)
            else:
                outputs = model(images)
            
            if isinstance(outputs, (list, tuple)):
                outputs = outputs[0]
            
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            
            pred = torch.argmax(outputs, dim=1, keepdim=True)
            metrics.update(pred, labels)
        
        avg_loss = total_loss / len(self.val_loader)
        computed_metrics = metrics.compute()
        
        return avg_loss, computed_metrics
    
    def save_checkpoint(self, epoch: int, stage: str, is_best: bool = False):
        """Save model checkpoint"""
        checkpoint = {
            'epoch': epoch,
            'stage': stage,
            'localizer_state': self.cascade.localizer.state_dict(),
            'segmentor_state': self.cascade.segmentor.state_dict(),
            'localizer_optimizer': self.localizer_optimizer.state_dict(),
            'segmentor_optimizer': self.segmentor_optimizer.state_dict(),
            'best_val_dice': self.best_val_dice,
            'config': self.config.__dict__
        }
        
        # Regular checkpoint
        checkpoint_path = Path(self.config.model_dir) / f'checkpoint_{stage}_epoch{epoch:03d}.pth'
        torch.save(checkpoint, checkpoint_path)
        
        # Best model
        if is_best:
            best_path = Path(self.config.model_dir) / f'best_model_{stage}.pth'
            torch.save(checkpoint, best_path)
            logger.info(f"Saved best model to {best_path}")
    
    def train(self, localizer_epochs: int = 100, segmentor_epochs: int = 200):
        """
        Full training pipeline for the cascade model.
        
        Args:
            localizer_epochs: Epochs for Stage A training
            segmentor_epochs: Epochs for Stage B training
        """
        logger.info("=" * 60)
        logger.info("STAGE A: TRAINING LOCALIZER")
        logger.info("=" * 60)
        
        for epoch in range(1, localizer_epochs + 1):
            train_loss, train_metrics = self.train_epoch_localizer(epoch)
            val_loss, val_metrics = self.validate('localizer')
            
            self.localizer_scheduler.step()
            lr = self.localizer_optimizer.param_groups[0]['lr']
            
            self.logger.log_epoch(epoch, train_loss, val_loss, train_metrics, val_metrics, lr)
            
            # Checkpointing
            is_best = val_metrics['mean_dice'] > self.best_val_dice
            if is_best:
                self.best_val_dice = val_metrics['mean_dice']
                self.patience_counter = 0
            else:
                self.patience_counter += 1
            
            if epoch % 20 == 0 or is_best:
                self.save_checkpoint(epoch, 'localizer', is_best)
            
            if self.patience_counter >= self.patience:
                logger.info(f"Early stopping at epoch {epoch}")
                break
        
        # Reset for segmentor training
        self.best_val_dice = 0.0
        self.patience_counter = 0
        
        logger.info("=" * 60)
        logger.info("STAGE B: TRAINING SEGMENTOR")
        logger.info("=" * 60)
        
        for epoch in range(1, segmentor_epochs + 1):
            train_loss, train_metrics = self.train_epoch_segmentor(epoch)
            val_loss, val_metrics = self.validate('segmentor')
            
            self.segmentor_scheduler.step()
            lr = self.segmentor_optimizer.param_groups[0]['lr']
            
            self.logger.log_epoch(epoch, train_loss, val_loss, train_metrics, val_metrics, lr)
            
            # Checkpointing
            is_best = val_metrics['mean_dice'] > self.best_val_dice
            if is_best:
                self.best_val_dice = val_metrics['mean_dice']
                self.patience_counter = 0
            else:
                self.patience_counter += 1
            
            if epoch % 20 == 0 or is_best:
                self.save_checkpoint(epoch, 'segmentor', is_best)
            
            if self.patience_counter >= self.patience:
                logger.info(f"Early stopping at epoch {epoch}")
                break
        
        self.logger.save_history()
        logger.info("Training complete!")


logger.info("Training engine initialized")

In [ ]:
# ============================================================================
# CELL 10: INFERENCE ENGINE
# ============================================================================

class InferenceEngine:
    """
    Production inference engine for HCC detection.
    
    Features:
    - Full cascade inference (Localizer → Segmentor)
    - Sliding window inference for large volumes
    - Post-processing and diagnostic report generation
    - Batch and single-volume inference support
    """
    
    def __init__(
        self,
        config: PipelineConfig,
        checkpoint_path: Optional[str] = None,
        device: torch.device = DEVICE
    ):
        self.config = config
        self.device = device
        
        # Initialize model
        self.cascade = CascadeSegmentationModel(config).to(device)
        
        # Load checkpoint if provided
        if checkpoint_path:
            self.load_checkpoint(checkpoint_path)
        
        self.cascade.eval()
        
        # Diagnostic processor
        self.diagnostic_processor = DiagnosticProcessor(
            malignancy_threshold=config.malignancy_threshold,
            voxel_spacing=config.target_spacing
        )
        
        logger.info("InferenceEngine initialized")
    
    def load_checkpoint(self, checkpoint_path: str):
        """Load model weights from checkpoint"""
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.cascade.localizer.load_state_dict(checkpoint['localizer_state'])
        self.cascade.segmentor.load_state_dict(checkpoint['segmentor_state'])
        logger.info(f"Loaded checkpoint from {checkpoint_path}")
    
    @torch.no_grad()
    def predict_volume(
        self,
        image: torch.Tensor,
        clinical: Optional[torch.Tensor] = None,
        use_sliding_window: bool = True
    ) -> Dict[str, Any]:
        """
        Run full inference pipeline on a single volume.
        
        Args:
            image: Preprocessed CT volume [1, 1, D, H, W]
            clinical: Optional clinical features [1, clinical_dim]
            use_sliding_window: Use sliding window for large volumes
            
        Returns:
            Dict with segmentation, probabilities, and diagnostic report
        """
        image = image.to(self.device)
        
        if use_sliding_window:
            # Stage A: Localizer with sliding window
            localizer_pred = sliding_window_inference(
                inputs=image,
                roi_size=self.config.localizer_roi_size,
                sw_batch_size=4,
                predictor=self.cascade.localizer,
                overlap=0.25
            )
            
            if isinstance(localizer_pred, list):
                localizer_pred = localizer_pred[0]
            
            # Stage B: Segmentor with sliding window
            if clinical is not None:
                clinical = clinical.to(self.device)
                has_clinical = torch.ones(1, dtype=torch.bool, device=self.device)
                
                def segmentor_predictor(x):
                    return self.cascade.segmentor(x, clinical, has_clinical)
            else:
                has_clinical = torch.zeros(1, dtype=torch.bool, device=self.device)
                
                def segmentor_predictor(x):
                    return self.cascade.segmentor(x, None, has_clinical)
            
            segmentor_pred = sliding_window_inference(
                inputs=image,
                roi_size=self.config.segmentor_roi_size,
                sw_batch_size=2,
                predictor=segmentor_predictor,
                overlap=0.5
            )
        else:
            # Direct inference (for smaller volumes)
            localizer_pred = self.cascade.localizer(image)
            if isinstance(localizer_pred, list):
                localizer_pred = localizer_pred[0]
            
            if clinical is not None:
                clinical = clinical.to(self.device)
                has_clinical = torch.ones(1, dtype=torch.bool, device=self.device)
            else:
                has_clinical = torch.zeros(1, dtype=torch.bool, device=self.device)
            
            segmentor_pred = self.cascade.segmentor(image, clinical, has_clinical)
        
        if isinstance(segmentor_pred, tuple):
            segmentor_pred = segmentor_pred[0]
        
        # Get probabilities and segmentation
        probs = F.softmax(segmentor_pred, dim=1)
        seg = torch.argmax(probs, dim=1)
        
        # Apply diagnostic logic
        seg_np = seg[0].cpu().numpy()
        probs_np = probs[0].cpu().numpy()
        
        diagnostic_result = self.diagnostic_processor.process_segmentation(seg_np, probs_np)
        report = self.diagnostic_processor.generate_report(diagnostic_result)
        
        return {
            'segmentation': seg_np,
            'probabilities': probs_np,
            'refined_segmentation': diagnostic_result['refined_segmentation'],
            'localizer_pred': torch.argmax(localizer_pred, dim=1)[0].cpu().numpy(),
            'diagnostic_result': diagnostic_result,
            'report': report
        }
    
    def process_nifti(
        self,
        image_path: str,
        clinical_data: Optional[Dict[str, float]] = None,
        save_output: bool = True,
        output_dir: Optional[str] = None
    ) -> Dict[str, Any]:
        """
        Process a single NIfTI file through the complete pipeline.
        
        Args:
            image_path: Path to input NIfTI volume
            clinical_data: Optional clinical features dict
            save_output: Save output segmentation
            output_dir: Output directory (default: config.output_dir)
            
        Returns:
            Complete inference results
        """
        import nibabel as nib
        
        # Load and preprocess
        transform = get_segmentor_transforms(self.config, is_train=False)
        
        data_dict = {'image': image_path, 'label': image_path}  # Use image as dummy label
        data = transform(data_dict)
        
        image = data['image'].unsqueeze(0)  # Add batch dimension
        
        # Process clinical data
        if clinical_data:
            processor = ClinicalDataProcessor()
            clinical = processor.process(clinical_data).unsqueeze(0)
        else:
            clinical = None
        
        # Run inference
        results = self.predict_volume(image, clinical)
        
        # Save outputs
        if save_output:
            output_dir = Path(output_dir or self.config.output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)
            
            input_name = Path(image_path).stem
            
            # Load original for header/affine
            original_nii = nib.load(image_path)
            
            # Save segmentation
            seg_nii = nib.Nifti1Image(
                results['refined_segmentation'].astype(np.int16),
                original_nii.affine,
                original_nii.header
            )
            seg_path = output_dir / f'{input_name}_segmentation.nii.gz'
            nib.save(seg_nii, seg_path)
            
            # Save report
            report_path = output_dir / f'{input_name}_report.txt'
            with open(report_path, 'w') as f:
                f.write(results['report'])
            
            logger.info(f"Saved segmentation to {seg_path}")
            logger.info(f"Saved report to {report_path}")
        
        return results


logger.info("Inference engine defined")

In [ ]:
# ============================================================================
# CELL 11: MAIN EXECUTION PIPELINE
# ============================================================================

def create_dataloaders(
    config: PipelineConfig,
    stage: str = 'segmentor'
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create MONAI CacheDataset loaders with patient-level splitting.
    
    Args:
        config: Pipeline configuration
        stage: 'localizer' or 'segmentor'
        
    Returns:
        Tuple of (train_loader, val_loader, test_loader)
    """
    # Discover data
    data_dicts = discover_nifti_files(config.data_paths)
    
    if len(data_dicts) == 0:
        logger.warning("No data found! Creating dummy data for demonstration...")
        # Create dummy data for testing pipeline
        data_dicts = [
            {'image': 'dummy.nii', 'label': 'dummy.nii', 'patient_id': f'patient_{i}'}
            for i in range(10)
        ]
    
    # Patient-level split (65/10/25)
    train_dicts, val_dicts, test_dicts = patient_level_split(
        data_dicts,
        train_ratio=config.train_ratio,
        val_ratio=config.val_ratio,
        test_ratio=config.test_ratio
    )
    
    # Get transforms
    if stage == 'localizer':
        train_transforms = get_localizer_transforms(config, is_train=True)
        val_transforms = get_localizer_transforms(config, is_train=False)
    else:
        train_transforms = get_segmentor_transforms(config, is_train=True)
        val_transforms = get_segmentor_transforms(config, is_train=False)
    
    # Create datasets with caching for memory efficiency
    train_ds = CacheDataset(
        data=train_dicts,
        transform=train_transforms,
        cache_rate=config.cache_rate,
        num_workers=config.num_workers,
        progress=True
    )
    
    val_ds = CacheDataset(
        data=val_dicts,
        transform=val_transforms,
        cache_rate=1.0,  # Cache all validation data
        num_workers=config.num_workers,
        progress=True
    )
    
    test_ds = CacheDataset(
        data=test_dicts,
        transform=val_transforms,
        cache_rate=1.0,
        num_workers=config.num_workers,
        progress=True
    )
    
    # Create dataloaders
    train_loader = MonaiDataLoader(
        train_ds,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        collate_fn=list_data_collate,
        pin_memory=True
    )
    
    val_loader = MonaiDataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=list_data_collate,
        pin_memory=True
    )
    
    test_loader = MonaiDataLoader(
        test_ds,
        batch_size=1,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=list_data_collate,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader


def main():
    """
    Main execution function for HCC Detection Pipeline.
    
    Workflow:
    1. Initialize configuration
    2. Discover and split data (patient-level)
    3. Train Stage A (Localizer)
    4. Train Stage B (Segmentor)
    5. Evaluate on test set
    6. Generate diagnostic reports
    """
    logger.info("=" * 80)
    logger.info("HEPATOCELLULAR CARCINOMA DETECTION PIPELINE")
    logger.info("Production-Grade 3D Medical Imaging for Liver Cancer Segmentation")
    logger.info("=" * 80)
    
    # Initialize configuration
    config = PipelineConfig()
    
    logger.info("\n📋 Pipeline Configuration:")
    logger.info(f"  • Data paths: {config.data_paths}")
    logger.info(f"  • HU Window: [{config.hu_min}, {config.hu_max}]")
    logger.info(f"  • Target spacing: {config.target_spacing} mm")
    logger.info(f"  • Batch size: {config.batch_size}")
    logger.info(f"  • Learning rate: {config.learning_rate}")
    logger.info(f"  • Max epochs: {config.max_epochs}")
    logger.info(f"  • Malignancy threshold: {config.malignancy_threshold * 100}%")
    
    # Create dataloaders
    logger.info("\n📦 Creating DataLoaders...")
    try:
        train_loader, val_loader, test_loader = create_dataloaders(config, stage='segmentor')
        logger.info(f"  • Train batches: {len(train_loader)}")
        logger.info(f"  • Val batches: {len(val_loader)}")
        logger.info(f"  • Test batches: {len(test_loader)}")
    except Exception as e:
        logger.warning(f"Could not create dataloaders: {e}")
        logger.info("Creating demonstration with synthetic data...")
        train_loader = None
        val_loader = None
        test_loader = None
    
    # Initialize trainer
    if train_loader is not None and val_loader is not None:
        logger.info("\n🏋️ Initializing Trainer...")
        trainer = CascadeTrainer(
            config=config,
            train_loader=train_loader,
            val_loader=val_loader,
            device=DEVICE
        )
        
        # Train the cascade
        logger.info("\n🚀 Starting Training...")
        trainer.train(
            localizer_epochs=min(100, config.max_epochs // 3),
            segmentor_epochs=min(200, config.max_epochs * 2 // 3)
        )
        
        # Save final model
        trainer.save_checkpoint(config.max_epochs, 'final', is_best=True)
    
    logger.info("\n✅ Pipeline execution complete!")
    logger.info(f"  • Models saved to: {config.model_dir}")
    logger.info(f"  • Logs saved to: {config.log_dir}")
    logger.info(f"  • Outputs saved to: {config.output_dir}")


# Architecture summary
def print_architecture_summary():
    """Print a summary of the model architecture"""
    config = PipelineConfig()
    
    print("\n" + "=" * 80)
    print("MODEL ARCHITECTURE SUMMARY")
    print("=" * 80)
    
    # Create models
    localizer = LiverLocalizer(
        in_channels=1,
        out_channels=2,
        channels=config.localizer_channels
    )
    
    segmentor = HybridResUNet(
        in_channels=1,
        out_channels=config.num_classes,
        channels=config.segmentor_channels,
        clinical_dim=10,
        use_clinical_fusion=True
    )
    
    print(f"\n📍 STAGE A - Liver Localizer (DynUNet)")
    print(f"   Input: 1 channel CT volume")
    print(f"   Output: 2 classes (Background, Liver)")
    print(f"   Channels: {config.localizer_channels}")
    print(f"   Parameters: {sum(p.numel() for p in localizer.parameters()):,}")
    
    print(f"\n🎯 STAGE B - Tumor Segmentor (HybridResUNet)")
    print(f"   Input: 1 channel CT volume + 10D clinical features")
    print(f"   Output: 3 classes (Background, Liver, Tumor)")
    print(f"   Channels: {config.segmentor_channels}")
    print(f"   Features: Residual blocks, SE attention, Clinical fusion")
    print(f"   Parameters: {sum(p.numel() for p in segmentor.parameters()):,}")
    
    total_params = sum(p.numel() for p in localizer.parameters()) + \
                   sum(p.numel() for p in segmentor.parameters())
    print(f"\n📊 Total Parameters: {total_params:,}")
    print("=" * 80)


# Run architecture summary
print_architecture_summary()

In [ ]:
# ============================================================================
# CELL 12: USAGE EXAMPLES & EXECUTION
# ============================================================================

# Example 1: Run full training pipeline
# Uncomment the following line to run training:
# main()

# Example 2: Single inference on a NIfTI file
def example_inference():
    """
    Example: Run inference on a single NIfTI file
    """
    config = PipelineConfig()
    
    # Initialize inference engine
    engine = InferenceEngine(
        config=config,
        checkpoint_path=None,  # Set to checkpoint path after training
        device=DEVICE
    )
    
    # Example clinical data (AFP levels, demographics, etc.)
    clinical_data = {
        'afp_level': 850.0,        # Elevated AFP (ng/mL)
        'age': 62,
        'gender': 1,               # Male
        'cirrhosis': 1,            # Cirrhosis present
        'hepatitis_b': 1,          # HBV positive
        'hepatitis_c': 0,          # HCV negative
        'bilirubin': 1.8,          # mg/dL
        'albumin': 3.2,            # g/dL
        'platelet_count': 120,     # 10^9/L
        'tumor_size_cm': 2.5,      # Estimated from imaging
    }
    
    # Run inference (replace with actual file path)
    # results = engine.process_nifti(
    #     image_path='/path/to/ct_scan.nii.gz',
    #     clinical_data=clinical_data,
    #     save_output=True
    # )
    
    # print(results['report'])
    
    logger.info("Inference example setup complete")
    return engine


# Example 3: Batch evaluation on test set
def example_evaluation():
    """
    Example: Evaluate model on test set with comprehensive metrics
    """
    config = PipelineConfig()
    
    # Create test dataloader
    _, _, test_loader = create_dataloaders(config, stage='segmentor')
    
    # Initialize inference engine with trained model
    engine = InferenceEngine(
        config=config,
        checkpoint_path=f'{config.model_dir}/best_model_segmentor.pth',
        device=DEVICE
    )
    
    # Evaluation metrics
    metrics = NPVDiceMetric(num_classes=3)
    
    all_results = []
    
    for batch in test_loader:
        image = batch['image'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        clinical = batch.get('clinical', None)
        
        # Run inference
        result = engine.predict_volume(
            image,
            clinical,
            use_sliding_window=True
        )
        
        # Update metrics
        pred = torch.from_numpy(result['refined_segmentation']).unsqueeze(0).unsqueeze(0)
        metrics.update(pred, label.cpu())
        
        all_results.append(result)
    
    # Compute final metrics
    final_metrics = metrics.compute()
    
    logger.info("=" * 60)
    logger.info("TEST SET EVALUATION RESULTS")
    logger.info("=" * 60)
    logger.info(f"Dice (Liver): {final_metrics['dice_liver']:.4f}")
    logger.info(f"Dice (Tumor): {final_metrics['dice_tumor']:.4f}")
    logger.info(f"Mean Dice: {final_metrics['mean_dice']:.4f}")
    logger.info(f"NPV (Tumor): {final_metrics['npv']:.4f}")
    logger.info("=" * 60)
    
    return final_metrics, all_results


# Quick demo with synthetic data
def quick_demo():
    """
    Quick demonstration of the pipeline with synthetic data
    """
    logger.info("=" * 60)
    logger.info("QUICK DEMO: HCC Detection Pipeline")
    logger.info("=" * 60)
    
    config = PipelineConfig()
    
    # Create cascade model
    cascade = CascadeSegmentationModel(config).to(DEVICE)
    
    # Create synthetic input
    batch_size = 1
    synthetic_ct = torch.randn(batch_size, 1, 64, 64, 64, device=DEVICE)
    synthetic_clinical = torch.randn(batch_size, 10, device=DEVICE)
    has_clinical = torch.ones(batch_size, dtype=torch.bool, device=DEVICE)
    
    logger.info(f"Input CT shape: {synthetic_ct.shape}")
    logger.info(f"Clinical features shape: {synthetic_clinical.shape}")
    
    # Forward pass
    cascade.eval()
    with torch.no_grad():
        outputs = cascade(
            synthetic_ct,
            clinical=synthetic_clinical,
            has_clinical=has_clinical,
            stage='both'
        )
    
    localizer_out = outputs['localizer_output']
    segmentor_out = outputs['segmentor_output']
    
    if isinstance(localizer_out, list):
        localizer_out = localizer_out[0]
    if isinstance(segmentor_out, tuple):
        segmentor_out = segmentor_out[0]
    
    logger.info(f"Localizer output shape: {localizer_out.shape}")
    logger.info(f"Segmentor output shape: {segmentor_out.shape}")
    
    # Apply post-processing
    refined_seg, diagnostic_results = apply_post_processing(segmentor_out, config)
    
    logger.info(f"Refined segmentation shape: {refined_seg.shape}")
    logger.info(f"Number of lesions detected: {diagnostic_results[0]['num_lesions_total']}")
    logger.info(f"Cancer detected: {diagnostic_results[0]['cancer_detected']}")
    
    # Generate report
    processor = DiagnosticProcessor(
        malignancy_threshold=config.malignancy_threshold,
        voxel_spacing=config.target_spacing
    )
    report = processor.generate_report(diagnostic_results[0])
    print("\n" + report)
    
    logger.info("\n✅ Quick demo complete!")
    return cascade, outputs


# Run quick demo
logger.info("Running quick demo...")
cascade_model, demo_outputs = quick_demo()

## 📚 Pipeline Documentation

### Architecture Layers Summary

| Layer | Component | Description |
|-------|-----------|-------------|
| **Layer 1** | Multimodal Ingestion | `MultimodalLiverDataset` handles .nii files + clinical bio-tabular vectors (AFP, demographics) |
| **Layer 2** | Precision Preprocessing | MONAI transforms: HU clipping [-75,150], 0.8mm isotropic, 3D augmentations (±40° rotation, flip, noise) |
| **Layer 3** | Hierarchical Cascade | Stage A: `LiverLocalizer` (DynUNet), Stage B: `HybridResUNet` with clinical fusion |
| **Layer 4** | Diagnostic Logic | 10% malignancy threshold + connected component analysis |
| **Layer 5** | Training & Metrics | Adam (lr=0.001), Dice+CE loss, DSC & NPV tracking |

### Key Features

- ✅ **Sub-centimeter lesion detection** via 0.8mm isotropic resampling
- ✅ **High NPV** for clinical triage (negative predictive value optimization)
- ✅ **Patient-level data splitting** (65/10/25) to prevent data leakage
- ✅ **Memory efficient** using MONAI CacheDataset
- ✅ **Mixed precision training** for faster GPU computation
- ✅ **Clinical data fusion** via attention-based late-stage integration
- ✅ **Deep supervision** for better gradient flow in deep networks

### Usage

```python
# Training (run on Kaggle with GPU)
main()

# Inference on a single file
engine = InferenceEngine(config, checkpoint_path='best_model.pth')
results = engine.process_nifti('ct_scan.nii.gz', clinical_data={'afp_level': 500, ...})
print(results['report'])
```

### Kaggle Dataset Paths
- `/kaggle/input/cbct-liver-and-liver-tumor-segmentation-train-data`
- `/kaggle/input/liver-tumor-segmentation-part-2`
- `/kaggle/input/liver-tumor-segmentation`

In [ ]:
# ============================================================================
# CELL 13: KAGGLE EXECUTION (Uncomment to run on Kaggle)
# ============================================================================

# Uncomment the following block to execute the full training pipeline on Kaggle:

"""
if __name__ == "__main__":
    # Verify GPU availability
    logger.info(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
        logger.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Run main training pipeline
    main()
    
    # Run evaluation on test set
    # metrics, results = example_evaluation()
"""

# To run training, execute:
# main()

logger.info("="*60)
logger.info("NOTEBOOK READY")
logger.info("All pipeline components have been defined successfully.")
logger.info("Run main() to start training, or use InferenceEngine for predictions.")
logger.info("="*60)